# Lecture 15 — Tensor-Network States (numpy companion)

A general $n$-qubit state has $2^n$ amplitudes, but *weakly entangled* states live on
a much smaller manifold. The key tool is the **Schmidt decomposition**: for any
bipartition $A|B$,
$$|\psi\rangle = \sum_{\alpha=1}^{\chi} \lambda_\alpha\,|a_\alpha\rangle_A\,|b_\alpha\rangle_B,$$
where the $\lambda_\alpha\ge 0$ are the singular values of the reshaped amplitude
matrix. The number of nonzero $\lambda_\alpha$ is the **Schmidt rank** $\chi$, and the
**entanglement entropy** is $S=-\sum_\alpha \lambda_\alpha^2\ln\lambda_\alpha^2$.

Sweeping the SVD across every cut turns $|\psi\rangle$ into a **matrix-product state
(MPS)** — a chain of rank-3 tensors. Truncating small singular values compresses the
state, with a controlled error equal to the tail of squared singular values.

This notebook is pure numpy (UnitaryLab's `Circuit` is used only to *produce* the
statevectors). We demonstrate:
- Schmidt rank / entropy for Bell, GHZ, and W states (GHZ entropy $=\ln 2$);
- exact MPS reconstruction of a 3-qubit state (error $<10^{-12}$);
- bond-dimension truncation error $=$ tail of squared singular values.

Contents:
1. Setup and helpers
2. Schmidt decomposition: Bell, GHZ, W
3. MPS via sequential SVD (exact reconstruction)
4. Truncation error vs. bond dimension
5. Your turn

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive")
from unitarylab.core import Circuit

np.set_printoptions(precision=4, suppress=True)

def statevector(build):
    '''Run a circuit-building callback and return its numpy statevector.'''
    qc = build()
    return qc.execute(backend="numpy", dtype=np.complex128).state

def schmidt(psi, n, cut):
    '''Singular values across the bipartition (first `cut` qubits | rest).'''
    M = psi.reshape(2**cut, 2**(n - cut))
    return np.linalg.svd(M, compute_uv=False)

def entanglement_entropy(svals, tol=1e-12):
    p = svals**2
    p = p[p > tol]
    return float(-np.sum(p * np.log(p)))

print("ready")

ready


## 2. Schmidt decomposition: Bell, GHZ, W

We build three canonical entangled states with `Circuit` and read off their Schmidt
spectra across the leading cut. The **Bell** and **GHZ** states are maximally
entangled across a single cut (two equal singular values $1/\sqrt2$, so entropy
$\ln 2$); the **W** state has a different, non-flat spectrum.

**Tolerance / claim:** GHZ has Schmidt rank exactly $2$ and entropy $=\ln 2$ (to
$10^{-12}$).

In [2]:
# Bell (2q): H, CX
bell = statevector(lambda: (lambda qc: (qc.h(0), qc.cx(0, 1), qc)[-1])(Circuit(2)))
# GHZ (3q): H, CX, CX
ghz = statevector(lambda: (lambda qc: (qc.h(0), qc.cx(0, 1), qc.cx(0, 2), qc)[-1])(Circuit(3)))
# W (3q): (|100>+|010>+|001>)/sqrt(3) — load explicitly
w = np.zeros(8, dtype=complex)
for idx in (0b001, 0b010, 0b100):
    w[idx] = 1 / np.sqrt(3)

for name, psi, n in [("Bell", bell, 2), ("GHZ", ghz, 3), ("W", w, 3)]:
    sv = schmidt(psi, n, 1)                    # cut: qubit 0 | rest
    sv = sv[sv > 1e-12]
    rank = len(sv)
    S = entanglement_entropy(sv)
    print(f"{name:4s}: Schmidt rank = {rank}, singular values = {np.round(sv, 4)}, "
          f"entropy = {S:.6f}")

# GHZ: exactly two singular values 1/sqrt(2), entropy ln 2.
sv_ghz = schmidt(ghz, 3, 1)
sv_ghz = sv_ghz[sv_ghz > 1e-12]
assert len(sv_ghz) == 2
assert abs(entanglement_entropy(sv_ghz) - np.log(2)) < 1e-12
print("\nVerified: GHZ Schmidt rank = 2 and entanglement entropy = ln 2 (to 1e-12).")

Bell: Schmidt rank = 2, singular values = [0.7071 0.7071], entropy = 0.693147
GHZ : Schmidt rank = 2, singular values = [0.7071 0.7071], entropy = 0.693147
W   : Schmidt rank = 2, singular values = [0.8165 0.5774], entropy = 0.636514

Verified: GHZ Schmidt rank = 2 and entanglement entropy = ln 2 (to 1e-12).


## 3. MPS via sequential SVD (exact reconstruction)

An MPS is obtained by repeatedly reshaping and SVD-ing the statevector from left to
right. With *no truncation* the factorization is exact, so contracting the MPS tensors
back together must reproduce the original amplitudes.

**Tolerance / claim:** reconstruction error $<10^{-12}$ for a random 3-qubit state.

In [3]:
def to_mps(psi, n):
    '''Left-canonical MPS as a list of rank-3 tensors A[k] with shape (chi_L, 2, chi_R).'''
    tensors = []
    bond = 1
    rem = psi.reshape(1, 2**n)
    for k in range(n - 1):
        rem = rem.reshape(bond * 2, 2**(n - 1 - k))
        U, s, Vh = np.linalg.svd(rem, full_matrices=False)
        chi = len(s)
        tensors.append(U.reshape(bond, 2, chi))
        rem = np.diag(s) @ Vh
        bond = chi
    tensors.append(rem.reshape(bond, 2, 1))
    return tensors

def from_mps(tensors):
    res = tensors[0]
    for T in tensors[1:]:
        res = np.tensordot(res, T, axes=([-1], [0]))
    return res.reshape(-1)

# Random normalized 3-qubit state.
rng = np.random.default_rng(0)
psi_rand = rng.standard_normal(8) + 1j * rng.standard_normal(8)
psi_rand /= np.linalg.norm(psi_rand)

mps = to_mps(psi_rand, 3)
print("MPS tensor shapes:", [T.shape for T in mps])
recon = from_mps(mps)
err = np.linalg.norm(recon - psi_rand)
print("reconstruction error (random 3q):", format(err, ".3e"))
assert err < 1e-12

# GHZ reconstruction as well.
err_ghz = np.linalg.norm(from_mps(to_mps(ghz, 3)) - ghz)
print("reconstruction error (GHZ):", format(err_ghz, ".3e"))
assert err_ghz < 1e-12
print("Verified: untruncated MPS reproduces the statevector to < 1e-12.")

MPS tensor shapes: [(1, 2, 2), (2, 2, 2), (2, 2, 1)]
reconstruction error (random 3q): 3.162e-16
reconstruction error (GHZ): 0.000e+00
Verified: untruncated MPS reproduces the statevector to < 1e-12.


## 4. Truncation error vs. bond dimension

Keeping only the largest $\chi$ singular values across a cut approximates the state.
The squared error of that rank-$\chi$ truncation equals the **tail of squared
singular values** $\sum_{\alpha>\chi}\lambda_\alpha^2$ (Eckart–Young). We verify this
identity on a random 4-qubit state at the central cut.

**Tolerance / claim:** measured truncation error$^2$ equals the discarded weight to
$10^{-12}$.

In [4]:
psi4 = rng.standard_normal(16) + 1j * rng.standard_normal(16)
psi4 /= np.linalg.norm(psi4)

M = psi4.reshape(4, 4)                       # central cut: qubits 0,1 | 2,3
U, s, Vh = np.linalg.svd(M, full_matrices=False)
print("central-cut singular values:", np.round(s, 4))

print("\n chi   trunc-err^2   tail sum   |diff|")
for chi in range(1, len(s) + 1):
    approx = (U[:, :chi] * s[:chi]) @ Vh[:chi, :]
    trunc_err_sq = np.linalg.norm(M - approx, "fro") ** 2
    tail = float(np.sum(s[chi:] ** 2))
    print(f"  {chi}    {trunc_err_sq:.3e}   {tail:.3e}   {abs(trunc_err_sq - tail):.1e}")
    assert abs(trunc_err_sq - tail) < 1e-12

print("\nVerified: rank-chi truncation error^2 = sum of discarded singular values^2 (to 1e-12).")

central-cut singular values: [0.8529 0.4466 0.2214 0.1549]

 chi   trunc-err^2   tail sum   |diff|
  1    2.725e-01   2.725e-01   1.7e-16
  2    7.301e-02   7.301e-02   2.8e-17
  3    2.399e-02   2.399e-02   6.9e-18
  4    3.428e-31   0.000e+00   3.4e-31

Verified: rank-chi truncation error^2 = sum of discarded singular values^2 (to 1e-12).


In [5]:
chis = np.arange(1, len(s) + 1)
tails = [float(np.sum(s[c:] ** 2)) for c in chis]
fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(chis, np.clip(tails, 1e-18, None), "o-")
ax.set_xlabel("retained bond dimension  $\\chi$")
ax.set_ylabel(r"discarded weight $\sum_{\alpha>\chi}\lambda_\alpha^2$")
ax.set_title("Truncation error vs. bond dimension (random 4-qubit state)")
ax.set_xticks(chis); ax.grid(True, which="both", ls=":")
plt.tight_layout(); plt.show()
print("keeping all 4 singular values gives zero truncation error (full rank at this cut).")

keeping all 4 singular values gives zero truncation error (full rank at this cut).


## 5. Your turn
- Compute the entanglement entropy of a random 4-qubit state at each of its three
  cuts; for a generic (volume-law) state the middle cut carries the most entropy.
- Build a product state (no entangling gates) and confirm every Schmidt rank is $1$
  and the MPS has bond dimension $1$ throughout — the classically-easy limit.
- Add a truncation step inside `to_mps` (drop singular values below a threshold) and
  measure the resulting global fidelity against the exact state; relate it to the sum
  of discarded weights across all cuts.

Lecture 16 uses these MPS ideas in **DMRG**, the variational workhorse for
one-dimensional ground states.